# Semana 05: Fundamentos Teóricos de Armazenamento em Nuvem e Segurança IAM

## Modelos de Armazenamento (EBS vs. EFS vs. S3) e Governança de Acesso com AWS IAM

Nesta aula consolidamos a **fundamentação teórica essencial** sobre os três modelos de armazenamento em nuvem e os mecanismos de segurança e identidade com o **AWS IAM (Identity and Access Management)**.

Entenderemos como os dados de uma planta industrial inteligente (**Fábrica Virtual Smart N1**) são armazenados, classificados e protegidos contra vazamentos e acessos indevidos antes de partirmos para implementações práticas em nuvem.

### Objetivos de Aprendizagem
- Diferenciar com precisão os 3 tipos de armazenamento em nuvem: **Bloco (EBS)**, **Arquivo (EFS)** e **Objeto (S3)**.
- Compreender a anatomia interna do **Amazon S3**: Buckets, Chaves (*Keys*), Metadados e o conceito de durabilidade de **11 noves (99.999999999%)**.
- Dominar os pilares do **AWS IAM**: Usuários, Grupos, Políticas (*Policies JSON*), Funções (*Roles*) e o **Princípio do Menor Privilégio**.
- Entender o risco crítico de **credenciais estáticas *hardcoded*** e por que as **IAM Roles** com credenciais temporárias são o padrão ouro de segurança.
- Executar simulações em Python para consolidar o funcionamento de um *Object Store* e do motor de avaliação de políticas IAM.

---


## 1. Modelos de Armazenamento em Nuvem: Bloco × Arquivo × Objeto

Nem todo dado tem as mesmas necessidades. O armazenamento em nuvem é dividido em três categorias fundamentais de acordo com a forma de acesso, performance e finalidade:

![Comparativo de Armazenamento em Nuvem](images/storage_comparativo_bloco_objeto_arquivo.jpg)

### 1.1 Armazenamento de Bloco (Block Storage — Ex: Amazon EBS)
- **Como funciona:** O armazenamento é dividido em blocos brutos de tamanho fixo. O sistema operacional trata esses blocos como um **disco rígido ou SSD virtual físico**.
- **Formatação:** Exige formatação com um sistema de arquivos (como `ext4`, `XFS` no Linux ou `NTFS` no Windows).
- **Características:** Altíssima velocidade, baixíssima latência (milissegundos) e acesso direto por apenas uma máquina por vez (em geral).
- **Casos de uso fabris:** Instalação do Sistema Operacional (disco de boot da EC2) e Bancos de Dados Relacionais com escrita intensa (PostgreSQL, MySQL, SQL Server).

### 1.2 Armazenamento de Arquivos (File Storage — Ex: Amazon EFS / NFS)
- **Como funciona:** Organiza os dados em uma árvore hierárquica tradicional de diretórios e arquivos (`/pastas/subpastas/arquivo.csv`).
- **Compartilhamento:** Permite que **centenas de servidores e instâncias acessem e editem os mesmos arquivos simultaneamente** via protocolos de rede como NFS ou SMB.
- **Casos de uso fabris:** Repositório compartilhado de documentações técnicas, esquemáticos CAD/CAM e diretório compartilhado de uploads para múltiplos servidores web.

### 1.3 Armazenamento de Objetos (Object Storage — Ex: Amazon S3)
- **Como funciona:** Os dados não ficam em pastas físicas nem em blocos de disco brutos. Cada arquivo é encapsulado como um **Objeto individual e imutável**, acompanhado de **Metadados descritivos** e uma **Chave de identificação única**.
- **Arquitetura Plana (*Flat Namespace*):** Não existem pastas reais no disco; as barras `/` nos nomes são apenas prefixos visuais.
- **Forma de Acesso:** Acessível via **Internet/API através de requisições HTTP REST** (`GET`, `PUT`, `DELETE`).
- **Escalabilidade e Custo:** Capacidade de armazenamento virtualmente infinita a uma fração do custo de discos EBS.
- **Casos de uso fabris:** *Data Lakes* de telemetria IoT de sensores, relatórios diários de produção, backups de longo prazo, logs e imagens de inspeção de qualidade.

### 1.4 Matriz Comparativa Rápida

| Critério | Armazenamento de Bloco (Amazon EBS) | Armazenamento de Arquivo (Amazon EFS) | Armazenamento de Objetos (Amazon S3) |
| :--- | :--- | :--- | :--- |
| **Analogia do Mundo Real** | Gaveta física com pastas organizadas por você | Pasta compartilhada de rede em um escritório | Guarda-volumes com bilhete numerado (*Ticket/Hash*) |
| **Unidade de Dado** | Blocos binários brutos | Arquivo em hierarquia de pastas | Objeto (Dados + Chave + Metadados) |
| **Protocolo de Acesso** | Barramento de disco (SCSI / NVMe) | Protocolo de rede (NFS v4, SMB) | Protocolo Web / API (HTTP / HTTPS REST) |
| **Acesso Concorrente** | 1 instância (geralmente exclusivo) | Centenas de instâncias simultâneas | Milhões de requisições web simultâneas |
| **Modificação de Dados** | Permite alterar 1 byte específico no meio do arquivo | Permite edição e append no arquivo | **Imutável**: para alterar, regrava o objeto inteiro |
| **Custo Relativo** | Médio / Alto por GB provisionado | Alto por GB utilizado | **Baixíssimo** por GB armazenado |

---


## 2. Anatomia e Funcionamento do Amazon S3

O **Amazon S3 (Simple Storage Service)** é o serviço de armazenamento de objetos mais utilizado do mundo. Vamos entender sua estrutura interna:

![Anatomia do Amazon S3](images/aws_s3_anatomia_objeto.jpg)

### 2.1 Os Elementos de um Objeto no S3

1. **Bucket (Balde/Contêiner):** É o contêiner raiz onde os objetos são guardados. 
   - **Regra de Ouro:** O nome do Bucket é **único globalmente em toda a AWS mundial**. Ninguém em nenhuma parte do mundo pode criar um bucket com o mesmo nome que o seu.
   - *Exemplo:* `smartn1-telemetria-producao-2026`.

2. **Key (Chave / Caminho do Objeto):** É o identificador textual único do objeto dentro do bucket.
   - *Exemplo:* `sensores/linha_prensa/temperatura/2026-09-01.json`.
   - Embora pareça uma estrutura de pastas, trata-se de uma chave de texto corrida (*String*).

3. **Value / Data (Dados / Conteúdo):** É a sequência de bytes do arquivo (pode ser texto, JSON, imagem, áudio, vídeo, binário de CLP). Pode variar de 0 bytes até **5 Terabytes** por objeto.

4. **Metadata (Metadados):** Pares chave-valor com informações sobre o arquivo:
   - **Metadados de Sistema:** Criados pela AWS automaticamente (`Content-Type`, `Content-Length`, `Last-Modified`, `ETag/MD5 Hash`).
   - **Metadados de Usuário (Custom):** Etiquetas personalizadas adicionadas pela fábrica (ex: `maquina-origem: torno-cnc-02`, `operador: turno-a`, `linha: estamparia`).

5. **Version ID (Identificador de Versão):** Quando o versionamento está ativo, sobrescrever um arquivo gera uma nova versão preservando o histórico completo contra deleções acidentais.

### 2.2 O que são os 11 Noves de Durabilidade (99.999999999%)?

- **Disponibilidade (Availability):** Capacidade do sistema estar no ar para responder requisições (ex: 99.9% de uptime ao ano).
- **Durabilidade (Durability):** Garantia de que o dado **NÃO será corrompido ou perdido fisicamente** nos discos da AWS.
- **O que significa 99.999999999% na prática?** Se você armazenar **10.000.000 (dez milhões)** de arquivos no Amazon S3, a probabilidade estatística é perder apenas **1 único arquivo a cada 10.000 anos**!
- Isso é alcançado porque o S3 replica automaticamente cada objeto em no mínimo **3 Zonas de Disponibilidade (AZs) distintas** e independentes dentro da mesma região.

---


## 3. Segurança e Governança de Acesso com AWS IAM

O **AWS IAM (Identity and Access Management)** é o serviço central de segurança e controle de acessos da nuvem.

![Segurança IAM e Roles](images/aws_iam_pilares_seguranca.jpg)

### 3.1 Os 4 Pilares da Identidade no IAM

1. **Users (Usuários IAM):** Representam **pessoas físicas** (ex: o engenheiro André, o operador Carlos). Possuem credenciais de login no console (usuário + senha + MFA) ou chaves de acesso CLI.
2. **Groups (Grupos IAM):** Coleção de usuários (ex: Grupo `EngenhariaAutomacao`, Grupo `OperadoresTurno`). Permite atribuir permissões para várias pessoas de uma só vez.
3. **Roles (Funções IAM):** Identidade destinada a ser assumida por **serviços ou aplicações** (como a nossa máquina virtual EC2, um container Docker ou uma função Lambda). **Não possuem senhas nem chaves estáticas permanentes**.
4. **Policies (Políticas de Permissão JSON):** Documentos estruturados que definem formalmente o que pode ou não ser feito.

### 3.2 O Princípio do Menor Privilégio (*Least Privilege*)
> **Regra de Ouro da Segurança em Nuvem:** Conceda a qualquer usuário, máquina ou aplicação **estritamente as permissões mínimas necessárias** para executar sua tarefa, e nada mais.

Por padrão, no IAM, **TUDO É NEGADO (Implicit Deny)**. Uma ação só é executada se houver uma regra explícita de `Allow`.

### 3.3 Estrutura de uma Política IAM (JSON)

Uma política IAM responde a 4 perguntas:
- **Effect:** Permitir (`Allow`) ou Negar (`Deny`)?
- **Action:** Qual operação da AWS? (ex: `s3:GetObject`, `s3:PutObject`, `ec2:StartInstances`)
- **Resource:** Em qual recurso específico? (Identificado pelo ARN — *Amazon Resource Name*)
- **Condition:** Sob quais condições? (ex: apenas se o IP for da fábrica, apenas em horário comercial)

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "PermitirUploadTelemetriaFabril",
      "Effect": "Allow",
      "Action": [
        "s3:PutObject",
        "s3:GetObject"
      ],
      "Resource": "arn:aws:s3:::smartn1-telemetria-2026/sensores/*"
    }
  ]
}
```

### 3.4 Perigo Crítico: Chaves Hardcoded × O Padrão Ouro das IAM Roles (`LabRole`)

- **A prática errada e perigosa (Hardcoding):** Colocar chaves estáticas (`AWS_ACCESS_KEY_ID` e `AWS_SECRET_ACCESS_KEY`) diretamente no código Python ou em arquivos `.env` commitados no GitHub.
  - *Consequência:* Bots varrem repositórios públicos em segundos. Chaves vazadas geram invasões e contas milionárias em poucos minutos.
- **A prática correta (IAM Roles):** Atribuir uma **Role** (no laboratório chamada de `LabRole` / `LabInstanceProfile`) diretamente à máquina EC2.
  - O serviço **AWS STS (Security Token Service)** injeta credenciais temporárias que são rotacionadas automaticamente a cada poucas horas.
  - O código Python usa essas credenciais transparentemente, **sem precisar guardar nenhuma senha no código**.

---



## 4. Exercícios de Fixação e Consolidação Conceitual

Responda às questões a seguir com base nos conceitos estudados nesta aula teórica:

### Questão 1 (Modelos de Armazenamento)
A fábrica **Smart N1** precisa armazenar dois tipos de dados:
1. O banco de dados relacional de controle de ordens de serviço (que exige leituras e escritas instantâneas de baixíssima latência pelo sistema operacional);
2. O histórico de 5 anos de fotos térmicas de alta resolução das máquinas para auditoria e análises futuras de Machine Learning.

Indique qual tipo de armazenamento (**Amazon EBS** ou **Amazon S3**) deve ser utilizado para cada um dos dois casos e justifique com base em performance e custo.

---

### Questão 2 (Anatomia do S3 e Durabilidade)
Por que uma empresa não precisa se preocupar em comprar fitas de backup ou configurar servidores redundantes para evitar a perda de arquivos gravados no Amazon S3? O que o conceito de **11 noves de durabilidade** garante tecnicamente?

---

### Questão 3 (Segurança e Governança IAM)
Um desenvolvedor júnior da fábrica sugeriu colocar a chave estática de administrador da AWS (`AWS_ACCESS_KEY_ID` e `AWS_SECRET_ACCESS_KEY`) dentro do script Python de coleta de sensores que roda dentro da EC2.  
Explique quais são os **dois graves riscos de segurança** dessa abordagem e como o uso de **IAM Roles (`LabRole`)** resolve esse problema de forma elegante e segura.

---

### Questão 4 (Avaliação de Políticas JSON)
Analise o seguinte cenário: Um usuário possui uma política que concede `Allow` para `s3:*` em todos os recursos, mas está inserido em um grupo que possui uma política com `Deny` explícito para `s3:DeleteBucket`.  
Se esse usuário tentar executar o comando para deletar um bucket, a operação será permitida ou negada? Explique a regra de precedência de avaliação do IAM.
